# Análise temporal dos gastos

Decomposição sazonal e tendências ao longo do tempo.
Quero ver se tem padrão mensal (tipo gastar mais em dezembro) e
como a tendência se comporta.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

In [ ]:
df = pd.read_parquet("data/processed/transacoes_clean.parquet")
df_ibov = pd.read_parquet("data/processed/ibovespa_clean.parquet")
df_orc = pd.read_parquet("data/processed/orcamento_clean.parquet")

print(f"{len(df)} transações de {df['data'].min().date()} a {df['data'].max().date()}")

## Série mensal de gastos

In [ ]:
# agrupar por mês
df["ano_mes"] = df["data"].dt.to_period("M")
mensal = df.groupby("ano_mes").agg(
    total=("valor", "sum"),
    qtd=("valor", "count"),
).reset_index()

mensal["data"] = mensal["ano_mes"].dt.to_timestamp()
mensal = mensal.sort_values("data")

# média móvel 3 meses
mensal["mm3"] = mensal["total"].rolling(3).mean()
# média móvel 6 meses
mensal["mm6"] = mensal["total"].rolling(6).mean()

mensal.head(10)

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))

ax.bar(mensal["data"], mensal["total"], width=25, color="steelblue", alpha=0.6, label="Total mensal")
ax.plot(mensal["data"], mensal["mm3"], color="orange", linewidth=2, label="MM 3 meses")
ax.plot(mensal["data"], mensal["mm6"], color="red", linewidth=2, label="MM 6 meses")

ax.set_title("Gastos mensais")
ax.set_ylabel("R$")
ax.legend()
plt.tight_layout()
plt.savefig("outputs/serie_mensal.png", dpi=150, bbox_inches="tight")
plt.show()

## Decomposição sazonal

Vou usar seasonal_decompose do statsmodels.
Se não tiver instalado uso um approach manual.

In [ ]:
# tentando statsmodels
try:
    from statsmodels.tsa.seasonal import seasonal_decompose

    ts = mensal.set_index("data")["total"].asfreq("MS")
    ts = ts.fillna(ts.median())  # preencher buracos se tiver

    decomp = seasonal_decompose(ts, model="additive", period=12)

    fig, axes = plt.subplots(4, 1, figsize=(14, 10), sharex=True)
    decomp.observed.plot(ax=axes[0], title="Observado")
    decomp.trend.plot(ax=axes[1], title="Tendência")
    decomp.seasonal.plot(ax=axes[2], title="Sazonalidade")
    decomp.resid.plot(ax=axes[3], title="Resíduo")
    for ax in axes:
        ax.set_ylabel("R$")
    plt.tight_layout()
    plt.savefig("outputs/decomposicao_sazonal.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("ok, statsmodels funcionou")

except ImportError:
    print("statsmodels não instalado, pulando decomposição")
    print("pip install statsmodels se quiser rodar")

## Sazonalidade por mês (boxplot)

Mesmo sem statsmodels dá pra ver o padrão.

In [ ]:
df["mes"] = df["data"].dt.month
df["nome_mes"] = df["data"].dt.strftime("%b")

meses_ordem = ["Jan", "Feb", "Mar", "Apr", "May", "Jun",
               "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]

# gasto mensal por mês do ano
gasto_mes = df.groupby([df["data"].dt.year, df["data"].dt.month])["valor"].sum().reset_index()
gasto_mes.columns = ["ano", "mes", "total"]

fig, ax = plt.subplots(figsize=(12, 5))
bp_data = [gasto_mes[gasto_mes["mes"] == m]["total"].values for m in range(1, 13)]
ax.boxplot(bp_data, labels=meses_ordem)
ax.set_title("Distribuição de gastos por mês do ano")
ax.set_ylabel("R$")
plt.tight_layout()
plt.savefig("outputs/sazonalidade_boxplot.png", dpi=150, bbox_inches="tight")
plt.show()

## Gastos por categoria ao longo do tempo

In [ ]:
# top 5 categorias
top_cats = df.groupby("categoria")["valor"].sum().nlargest(5).index.tolist()

fig, ax = plt.subplots(figsize=(14, 6))

for cat in top_cats:
    cat_mensal = df[df["categoria"] == cat].groupby("ano_mes")["valor"].sum().reset_index()
    cat_mensal["data"] = cat_mensal["ano_mes"].dt.to_timestamp()
    ax.plot(cat_mensal["data"], cat_mensal["valor"], "o-", label=cat, markersize=4)

ax.set_title("Top 5 categorias ao longo do tempo")
ax.set_ylabel("R$")
ax.legend(bbox_to_anchor=(1.05, 1), loc="upper left", fontsize=9)
plt.tight_layout()
plt.savefig("outputs/categorias_tempo.png", dpi=150, bbox_inches="tight")
plt.show()

## Correlação gastos vs Ibovespa

In [ ]:
# ibov mensal
df_ibov["ano_mes"] = df_ibov["data"].dt.to_period("M")
ibov_mensal = df_ibov.groupby("ano_mes")["Close"].mean().reset_index()
ibov_mensal.columns = ["ano_mes", "ibov_media"]

# merge
merged = mensal.merge(ibov_mensal, on="ano_mes", how="inner")

corr = merged["total"].corr(merged["ibov_media"])
print(f"Correlação gastos vs ibovespa: {corr:.3f}")

In [ ]:
fig, ax1 = plt.subplots(figsize=(14, 5))

ax1.bar(merged["data"], merged["total"], width=25, alpha=0.5, color="steelblue", label="Gastos")
ax1.set_ylabel("Gastos (R$)", color="steelblue")

ax2 = ax1.twinx()
ax2.plot(merged["data"], merged["ibov_media"], color="red", linewidth=2, label="Ibovespa")
ax2.set_ylabel("Ibovespa (pts)", color="red")

ax1.set_title(f"Gastos vs Ibovespa (corr={corr:.2f})")
fig.legend(loc="upper left", bbox_to_anchor=(0.12, 0.88))
plt.tight_layout()
plt.savefig("outputs/gastos_vs_ibovespa.png", dpi=150, bbox_inches="tight")
plt.show()

## Orçamento vs Realizado

In [ ]:
df_orc["ano_mes"] = df_orc["Mês"].dt.to_period("M")
comp = mensal.merge(df_orc[["ano_mes", "Limite"]], on="ano_mes", how="inner")

if len(comp) > 0:
    comp["diff"] = comp["total"] - comp["Limite"]
    comp["estourou"] = comp["diff"] > 0

    fig, ax = plt.subplots(figsize=(14, 5))
    ax.bar(comp["data"], comp["total"], width=25, alpha=0.6, color="steelblue", label="Real")
    ax.plot(comp["data"], comp["Limite"], "ro-", markersize=5, label="Orçamento")

    # pintar os meses estourados de vermelho
    for _, row in comp[comp["estourou"]].iterrows():
        ax.bar(row["data"], row["total"], width=25, alpha=0.6, color="salmon")

    ax.set_title(f"Orçamento vs Realizado ({comp['estourou'].sum()} meses estourados)")
    ax.set_ylabel("R$")
    ax.legend()
    plt.tight_layout()
    plt.savefig("outputs/orcamento_vs_real.png", dpi=150, bbox_inches="tight")
    plt.show()
else:
    print("sem interseção entre orçamento e transações")

## Dashboard resumo (texto)

Resumo geral dos indicadores.

In [ ]:
total_periodo = df["valor"].sum()
media_mensal = mensal["total"].mean()
mes_mais_caro = mensal.loc[mensal["total"].idxmax()]
mes_mais_barato = mensal.loc[mensal["total"].idxmin()]
cat_top = df.groupby("categoria")["valor"].sum().nlargest(3)
anom = df["anomalia"].sum() if "anomalia" in df.columns else 0

print("=" * 50)
print("RESUMO FINANCEIRO")
print("=" * 50)
print(f"Período:       {df['data'].min().date()} a {df['data'].max().date()}")
print(f"Total:         R$ {total_periodo:,.2f}")
print(f"Média mensal:  R$ {media_mensal:,.2f}")
print(f"Mês mais caro: {mes_mais_caro['ano_mes']} (R$ {mes_mais_caro['total']:,.2f})")
print(f"Mês mais leve: {mes_mais_barato['ano_mes']} (R$ {mes_mais_barato['total']:,.2f})")
print(f"Anomalias:     {anom}")
print(f"\nTop categorias:")
for cat, val in cat_top.items():
    print(f"  {cat:<20s} R$ {val:,.2f}")
print("=" * 50)

In [ ]:
# TO DO: testar ARIMA / Prophet depois com mais dados
# por enquanto os 2 anos não dão período suficiente pra forecasting automático funcionar bem